Whats is persistance ?
Suppose we are doing chat on our chatbot and by mistake we exited olr we shut down the bot and tommorow again we need to use the same chat forn sa,e relenac so we could access it due to persisitance.

Benefit of persistnce?
Yes the imp one is fault tolernace :-
IF in our bot some system fails so it will use fallback staregy and other things could be worked


In [11]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()
llm =  ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [7]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanantion:str
    

In [8]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}


In [9]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}


In [12]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)


In [ ]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)
